# ALEX 구현 실습 — 2026-09-25 에 들어간 기능으로 공부하기

목표: 오늘 ALEX 에 들어간 기능을 **작은 함수로 직접 만들어 보며** 이해합니다.
데스크톱 전용 설정, 빈 답의 원인 판단, 배운 것(교훈) 고르기, 자동 학습, 스스로 검색, 검색 캐시, 도구 호출 루프, 코드 지문을 다룹니다.

- 위에서 아래로 `Shift+Enter` 를 누르세요. `TODO` 함수의 `raise NotImplementedError` 를 본인 코드로 바꾼 뒤 바로 아래 **검사 셀**을 실행하세요. 처음에 `아직 풀지 않았습니다` 나 `FAIL` 이 나오는 것은 정상입니다.
- 모델을 부르거나 인터넷·파일·대화 기록을 건드리지 않습니다. **관찰** 셀만 실제 ALEX 코드를 *읽어서* 비교합니다.
- 막히면 [ALEX_IMPLEMENTATION_LAB_HINTS.md](ALEX_IMPLEMENTATION_LAB_HINTS.md) 의 힌트를 보세요. 정답 코드는 없습니다. 다 풀면 각 문제의 **실제 코드 위치**를 열어 비교해 보세요.
- 난이도: ★ 기초 · ★★ 중간 · ★★★ 어려움. 1→14 순서대로 풀면 앞 문제의 함수를 뒤에서 다시 씁니다(5→6).

In [ ]:
from pathlib import Path
import sys, math, re, json, hashlib
from datetime import datetime, timedelta, timezone

project_root = Path.cwd().resolve().parent
HAVE_ALEX = (project_root / 'jarvis' / 'lessons.py').is_file()
if HAVE_ALEX:
    sys.path.insert(0, str(project_root))
print('실제 ALEX 코드 관찰:', '가능' if HAVE_ALEX else '불가 — study 폴더에서 JupyterLab 을 시작하면 관찰 셀도 쓸 수 있습니다')

def check(name, cases):
    """cases: [(설명, 검사 함수), ...]. 하나라도 틀리면 어떤 경우가 틀렸는지 알려 줍니다."""
    for number, (label, test) in enumerate(cases, start=1):
        try:
            ok = test()
        except NotImplementedError:
            print(f'{name}: 아직 풀지 않았습니다 (TODO)')
            return
        except Exception as exc:
            print(f'{name}: FAIL — {number}번 검사 「{label}」 에서 {type(exc).__name__}: {exc}')
            return
        if not ok:
            print(f'{name}: FAIL — {number}번 검사 「{label}」')
            return
    print(f'{name}: PASS ({len(cases)}/{len(cases)})')

def observe(title, func):
    """실제 ALEX 코드를 읽기만 합니다. 다른 사람이 코드를 고치는 중이라 실패해도 실습에는 영향이 없습니다."""
    if not HAVE_ALEX:
        print(title, ': ALEX 코드를 찾지 못해 건너뜁니다')
        return
    try:
        func()
    except Exception as exc:
        print(title, ': 관찰하지 못했습니다 —', type(exc).__name__, exc)

---
## 1부. 기초 ★

### 1. 프로필에 따라 Ollama 환경 만들기 ★ (ADR-054)

**배경:** 사용자가 고른 `qwen3.5:122b` 가 RTX 5080 에서 플래시 어텐션(FA) 커널 오류로 죽었습니다. 그래서 **데스크톱에서만** FA 를 끕니다. 노트북은 FA 를 끄고 잰 적이 없으니 그대로 둡니다 — *재지 않은 조건으로 바꾸지 않는다*.

`ollama_environment(profile, base_env)` 는 `base_env` 를 **복사한 새 사전**을 돌려줍니다.
- 항상: `"OLLAMA_HOST": "127.0.0.1:11434"`, `"OLLAMA_NO_CLOUD": "1"` (원래 값이 있어도 덮어씁니다)
- `profile` 이 `"DESKTOP"` 이면 `"OLLAMA_FLASH_ATTENTION": "0"` 추가
- `base_env` 원본은 **바꾸지 않습니다.**

실제 코드: `jarvis/launcher.py` 의 `ollama_environment`

In [ ]:
def ollama_environment(profile: str, base_env: dict) -> dict:
    # TODO: base_env 를 복사한 뒤 필요한 키를 넣어 돌려주세요.
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
def _unchanged_after_call():
    base = {'PATH': r'C:\Windows', 'OLLAMA_NO_CLOUD': '0'}
    ollama_environment('DESKTOP', base)
    return base == {'PATH': r'C:\Windows', 'OLLAMA_NO_CLOUD': '0'}

check('1. 프로필별 환경', [
    ('데스크톱은 FA 를 끈다', lambda: ollama_environment('DESKTOP', {})['OLLAMA_FLASH_ATTENTION'] == '0'),
    ('노트북은 FA 키가 없다', lambda: 'OLLAMA_FLASH_ATTENTION' not in ollama_environment('LAPTOP', {})),
    ('어느 프로필이든 클라우드 금지', lambda: all(ollama_environment(p, {'OLLAMA_NO_CLOUD': '0'})['OLLAMA_NO_CLOUD'] == '1'
                                         for p in ('DESKTOP', 'LAPTOP', 'UNKNOWN'))),
    ('주소는 127.0.0.1 만', lambda: ollama_environment('LAPTOP', {})['OLLAMA_HOST'] == '127.0.0.1:11434'),
    ('원래 있던 키는 남긴다', lambda: ollama_environment('LAPTOP', {'PATH': 'x'})['PATH'] == 'x'),
    ('원본 사전은 바뀌지 않는다', _unchanged_after_call),
])

In [ ]:
# 관찰: 실제 ALEX 는 프로필마다 FA 를 어떻게 둘까요? (읽기만 합니다)
def _look():
    from jarvis.launcher import ollama_environment as real
    from jarvis.hardware import PerformanceProfile
    for profile in PerformanceProfile:
        print(f'{profile.value:8} → OLLAMA_FLASH_ATTENTION =', real(profile).get('OLLAMA_FLASH_ATTENTION', '(Ollama 기본값)'))
observe('관찰 1', _look)

### 2. 몇 단어가 겹쳐야 "관련 있다" 고 볼까 ★ (ADR-055)

ALEX 는 배운 것(교훈) 가운데 지금 질문과 관련된 것만 프롬프트에 넣습니다. 질문의 뜻 있는 단어 가운데 **절반**(최소 2개)이 겹쳐야 관련 있다고 봅니다. 질문이 아주 길어도 요구치가 끝없이 커지지 않게 **10단어에서 멈춥니다.**

`required_matches(n) = max(2, 올림(0.5 × min(n, 10)))` — `math.ceil` 을 쓰세요.

실제 코드: `jarvis/lessons.py` 의 `required_matches`

In [ ]:
def required_matches(n_words: int) -> int:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
check('2. 필요한 겹침 수', [
    (f'{n}단어 → {want}', (lambda n=n, want=want: required_matches(n) == want))
    for n, want in [(1, 2), (3, 2), (4, 2), (5, 3), (7, 4), (10, 5), (30, 5)]
])

### 3. 빈 답은 왜 났을까 ★ (PERFORMANCE 26-1)

**실제 사건:** 문맥 창 8192 토큰에 긴 대화가 7830 토큰을 차지해, 모델이 남은 361 토큰을 생각(thinking)에 다 쓰고 **본문 0자**를 냈습니다. 앱은 "모델이 없다, Ollama 를 확인하라" 고 엉뚱하게 안내했습니다. 이제는 원인을 구분합니다.

`exhaustion_cause(done_reason, prompt_tokens, num_predict, num_ctx)`
- `done_reason` 이 `"length"` 가 아니면 `None` (한도 때문이 아님)
- `prompt_tokens` 가 정수이고, `num_ctx` 가 0 이 아니고, `prompt_tokens + num_predict > num_ctx` 이면 `"context"` (문맥 창이 참)
- 그 밖에는 `"output"` (출력 한도를 생각에 다 씀)

실제 코드: `jarvis/providers.py` 의 `_raise_if_exhausted`

In [ ]:
def exhaustion_cause(done_reason, prompt_tokens, num_predict: int, num_ctx: int):
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
check('3. 빈 답의 원인', [
    ('그날의 사건: 7830 + 6144 > 8192', lambda: exhaustion_cause('length', 7830, 6144, 8192) == 'context'),
    ('프롬프트가 짧으면 출력 한도 탓', lambda: exhaustion_cause('length', 1000, 6144, 16384) == 'output'),
    ('정상 종료면 한도 문제가 아니다', lambda: exhaustion_cause('stop', 7830, 6144, 8192) is None),
    ('프롬프트 토큰을 모르면 출력 한도 탓', lambda: exhaustion_cause('length', None, 6144, 8192) == 'output'),
    ('num_ctx 0(자동)이면 판단하지 않는다', lambda: exhaustion_cause('length', 7830, 6144, 0) == 'output'),
    ('16384 로 올린 뒤: 7830 + 6144 < 16384', lambda: exhaustion_cause('length', 7830, 6144, 16384) == 'output'),
])

### 4. 출처에 번호 매기기 ★ (ADR-056)

ALEX 가 스스로 검색하면 출처는 모델이 아니라 **앱이** 답 끝에 붙입니다(모델은 번호를 빠뜨리거나 지어낼 수 있으니까요). 같은 주소는 같은 번호를 씁니다.

`source_number(sources, title, url)`: `sources` 는 `[(제목, 주소), ...]` 목록입니다. `url` 이 이미 있으면 그 번호(**1부터**)를 돌려주고, 없으면 끝에 `(title, url)` 을 추가하고 새 번호를 돌려줍니다. 목록은 제자리에서 바뀝니다.

실제 코드: `jarvis/core.py` 의 `_tool_source`

In [ ]:
def source_number(sources: list, title: str, url: str) -> int:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
def _numbering():
    s = []
    got = [source_number(s, 'A', 'https://a'), source_number(s, 'B', 'https://b'),
           source_number(s, 'A 다시', 'https://a'), source_number(s, 'C', 'https://c')]
    return got == [1, 2, 1, 3] and s == [('A', 'https://a'), ('B', 'https://b'), ('C', 'https://c')]

check('4. 출처 번호', [
    ('첫 출처는 1번', lambda: source_number([], 'A', 'https://a') == 1),
    ('같은 주소는 같은 번호, 목록도 맞게', _numbering),
])

---
## 2부. 문자열과 정규식 ★★

### 5. 한글 단어를 두 글자 조각으로 ★★ (ADR-055)

"인터넷**은**" 과 "인터넷**을**" 은 단어로는 다르지만 두 글자 조각(`인터`·`터넷`)은 겹칩니다. 한국어는 조사·어미가 붙어 모양이 바뀌므로 ALEX 는 조각으로 비교합니다.

`pieces(word)`: 첫 글자가 한글(`'가' <= ch <= '힣'`)이면 두 글자씩 **겹쳐** 자른 목록, 아니면(영문·숫자) `[word]` 그대로. 단어는 2글자 이상이라고 가정합니다.

실제 코드: `jarvis/lessons.py` 의 `_pieces`

In [ ]:
def pieces(word: str) -> list:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
check('5. 두 글자 조각', [
    ('인터넷은', lambda: pieces('인터넷은') == ['인터', '터넷', '넷은']),
    ('두 글자 단어', lambda: pieces('코드') == ['코드']),
    ('영문은 그대로', lambda: pieces('rtx') == ['rtx']),
    ('숫자도 그대로', lambda: pieces('5090') == ['5090']),
])

### 6. 질문에서 뜻 있는 단어 고르기 ★★ (ADR-055)

`query_words(text, stop)`:
1. `text.lower()` 에서 정규식 `r"[가-힣]{2,}|[0-9a-z_]{2,}"` 로 단어를 모두 찾습니다(`re.findall`).
2. 각 단어를 5번의 `pieces()` 로 자릅니다.
3. **첫 조각**이 `stop`(불용어 집합)에 있으면 그 단어는 뺍니다 — "해줘", "알려줘", "지금" 처럼 어느 질문에나 나오는 말.
4. **첫 조각(어간)이 같은 단어는 처음 것 하나만** 남깁니다 — "설정한" 과 "설정한거야" 를 두 번 세면 흔한 말 하나로 기준을 넘어 버립니다(실제로 그랬습니다).

돌려주는 값: 조각 목록들의 목록(나온 순서). 힌트: 어간을 키로 하는 사전.

실제 코드: `jarvis/lessons.py` 의 `query_words`

In [ ]:
STOP = {'해줘', '지금', '알려', '려줘'}

def query_words(text: str, stop: set) -> list:
    # TODO: 5번의 pieces() 를 쓰세요.
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
check('6. 뜻 있는 단어', [
    ('어간이 같으면 한 번만', lambda: query_words('설정한 모델 설정한거야', STOP) == [['설정', '정한'], ['모델']]),
    ('영문·숫자·불용어', lambda: query_words('RTX 5090 가격 알려줘', STOP) == [['rtx'], ['5090'], ['가격']]),
    ('불용어뿐이면 빈 목록', lambda: query_words('지금 해줘', STOP) == []),
    ('한 글자 단어는 없다', lambda: query_words('왜 안 돼', STOP) == []),
])

In [ ]:
# 관찰: 실제 ALEX 가 고르는 단어 (ALEX 는 불용어가 훨씬 많습니다)
def _look():
    from jarvis.lessons import query_words as real
    for q in ['설정한 모델 설정한거야', '자연어로도 처리가 될거애', 'RTX 5090 가격 알려줘']:
        print(q, '→', real(q))
observe('관찰 6', _look)

### 7. "틀렸어" 알아채기 ★★ (ADR-057)

사용자가 직전 답을 바로잡는 말이면 ALEX 는 그 말을 **그대로** 교훈으로 배웁니다(판단한 사람이 사용자이니 그 자체가 승인). 규칙(실습용 간단판):

1. 공백을 한 칸으로 정리한 문장이 **6글자보다 짧으면** `False`
2. `'내가 틀'` 이 있으면 `False` — "내가 틀렸나?" 는 자기 의심입니다
3. `'틀렸'` 또는 `'그게 아니라'` 가 있으면 `True`
4. `'텐데?'` 가 있으면 `True` — "다를텐데?"
5. 문장이 `'잖아'` 또는 `'잖아요'` 로 **끝나면** `True` — 끝에 `. ? ! … ~` 가 붙어도 됩니다. 정규식 `r'잖아요?[.?!…~]*$'`
6. 그 밖에는 `False`

실제 코드: `jarvis/lessons.py` 의 `looks_like_correction` (실제는 규칙이 조금 더 많습니다)

In [ ]:
def looks_like_correction(text: str) -> bool:
    # TODO: ' '.join(text.split()) 로 공백부터 정리하세요.
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
YES = ['틀렸어, qwen3.8은 8월에 나왔어', '그게 아니라 16GB 야', '브라우저 검색 되잖아..',
       '지금 데스크탑 모델은 다를텐데? 노트북 기준이야??', '인터넷은 되잖아요!']
NO = ['내가 틀렸나?', '고마워', '어떻게 하는데?', '코드 수정됐을텐데 다시 확인해봐', '잖아', '파이썬 리스트 뒤집는 법']
check('7. 바로잡는 말', [(f'True: {t}', (lambda t=t: looks_like_correction(t) is True)) for t in YES]
                     + [(f'False: {t}', (lambda t=t: looks_like_correction(t) is False)) for t in NO])

### 8. 개인 정보 같은 검색어 거르기 ★★ (ADR-056)

ALEX 가 스스로 검색할 때 **밖으로 나가는 것은 검색어뿐**입니다. 그런데 검색어는 모델이 지으니, 개인 정보 모양이면 보내지 않습니다.

`looks_private(query)`: 아래 중 하나라도 있으면 `True`
- 메일 주소 (`이름@도메인.점` 모양)
- Windows 경로 (`C:\` 처럼 영문 한 글자 + `:\`)
- `/home/` 또는 `/Users/` 가 들어간 경로
- **9자리 이상** 이어진 숫자 (전화·계좌 따위) — 4자리 연도나 모델 번호는 괜찮습니다

실제 코드: `jarvis/core.py` 의 `PRIVATE_QUERY`

In [ ]:
def looks_private(query: str) -> bool:
    # TODO: re.search 한 번으로도, 조건 여러 개로도 됩니다.
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
check('8. 개인 정보 검색어', [
    ('메일', lambda: looks_private('user@example.com 비밀번호') is True),
    ('Windows 경로', lambda: looks_private(r'C:\Users\alexuser\secret.txt 내용') is True),
    ('리눅스 경로', lambda: looks_private('/home/heeschoi/data 폴더') is True),
    ('긴 숫자', lambda: looks_private('계좌 123456789012') is True),
    ('평범한 검색어', lambda: looks_private('Qwen3.8 release date') is False),
    ('연도·모델 번호는 괜찮다', lambda: looks_private('RTX 5090 가격 2026') is False),
])

---
## 3부. 자료구조와 정렬 ★★

### 9. 이 단어가 경험에 나왔다고 볼까 ★★ (ADR-055)

`word_found(pieces, terms, stop)`:
- **어간(첫 조각)이 `terms` 에 반드시** 있어야 합니다.
- 그리고 `stop` 에 없는 조각들 가운데 **절반 이상**이 `terms` 에 있어야 `True` 입니다.

**실제 사건:** 질문 "갑자기 코드가 왜나와" 의 '갑자기' 가, 교훈 속 '**자기** 앞선 답' 의 '자기' 조각과 겹쳐 엉뚱한 교훈이 붙었습니다. 어간 '갑자' 가 없으니 `False` 여야 합니다.

실제 코드: `jarvis/lessons.py` 의 `_word_found`

In [ ]:
def word_found(pieces: list, terms: set, stop: set) -> bool:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
check('9. 단어가 나왔나', [
    ('갑자기 ≠ 자기', lambda: word_found(['갑자', '자기'], {'자기', '코드'}, set()) is False),
    ('코드가: 어간만 있어도 절반', lambda: word_found(['코드', '드가'], {'코드'}, set()) is True),
    ('인터넷은: 3조각 중 2개', lambda: word_found(['인터', '터넷', '넷은'], {'인터', '터넷'}, set()) is True),
    ('인터넷은: 3조각 중 1개', lambda: word_found(['인터', '터넷', '넷은'], {'인터'}, set()) is False),
    ('영문 단어', lambda: word_found(['rtx'], {'rtx'}, set()) is True),
    ('불용어 조각은 세지 않는다', lambda: word_found(['설정', '정한', '한거', '거야'], {'설정', '정한'}, {'거야'}) is True),
])

### 10. 배운 것 고르기 ★★★ (ADR-055·057)

`lessons` 는 사전 목록입니다: `{'id', 'kind', 'stage', 'source', 'created_at'(datetime), 'score'(겹친 단어 수)}`.

`pick_lessons(lessons, needed, limit, now)`:
1. **쓸 수 있는 것만**: `kind == 'answer'` 이고, `stage == 'approved'` 이거나 (`stage == 'candidate'` 이고 `source` 가 `'found:'` 로 시작)
   — 사용자가 바로잡은 것은 approved, ALEX 가 웹에서 찾은 사실은 candidate 로 들어옵니다.
2. `source` 가 `'found:'` 로 시작하는 것은 `created_at` 이 `now` 보다 **90일 넘게** 전이면 뺍니다 — 웹에서 찾은 것은 오래되면 틀릴 수 있으니까요. 바로잡은 것은 만료가 없습니다.
3. `score >= needed` 인 것만
4. `score` 큰 순, 같으면 `created_at` **최근** 순으로 정렬 (힌트: `sorted(..., key=lambda x: (x['score'], x['created_at']), reverse=True)`)
5. 앞에서 `limit` 개의 `id` 목록을 돌려줍니다.

실제 코드: `jarvis/lessons.py` 의 `usable`, `_expired`, `select_lessons`

In [ ]:
def pick_lessons(lessons: list, needed: int, limit: int, now: datetime) -> list:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
_now = datetime(2026, 9, 25, 21, 0, tzinfo=timezone.utc)
def _L(id, kind='answer', stage='approved', source='correction:s1', days_ago=1, score=3):
    return {'id': id, 'kind': kind, 'stage': stage, 'source': source,
            'created_at': _now - timedelta(days=days_ago), 'score': score}
POOL = [
    _L('a', score=3, days_ago=1),                                  # 바로잡은 것
    _L('b', stage='raw', score=5),                                 # 검토 전
    _L('c', stage='candidate', source='found:q1', score=4),        # 웹에서 찾은 것
    _L('d', stage='candidate', source='session:x', score=4),       # 찾은 것이 아닌 후보
    _L('e', source='found:q2', score=4, days_ago=100),             # 오래된 찾은 것
    _L('f', score=3, days_ago=200),                                # 오래된 바로잡은 것(만료 없음)
    _L('g', kind='identity', score=5),                             # 종류가 다름
    _L('h', score=1),                                              # 덜 겹침
]
check('10. 배운 것 고르기', [
    ('고르고 정렬하기', lambda: pick_lessons(POOL, 2, 10, _now) == ['c', 'a', 'f']),
    ('limit', lambda: pick_lessons(POOL, 2, 2, _now) == ['c', 'a']),
    ('기준을 올리면', lambda: pick_lessons(POOL, 4, 10, _now) == ['c']),
    ('빈 목록', lambda: pick_lessons([], 2, 3, _now) == []),
])

---
## 4부. 설계 패턴 ★★★

### 11. 검색엔진을 괴롭히지 않는 캐시 ★★★ (ADR-056)

**실제 사건:** 스스로 검색을 시험하며 몇 분 사이 15번쯤 검색하자 DuckDuckGo 가 자동 요청으로 막았습니다. 막힌 채 계속 보내면 차단이 길어집니다.

`SearchCache(fetch, clock)`:
- `fetch(query)` 는 결과 목록을 돌려주거나 `Blocked` 를 던집니다.
- `clock()` 은 지금 시각(초)을 돌려줍니다. **시계를 밖에서 받는** 이유: 테스트에서 10분을 기다리지 않고 시간을 움직이려고요.

`search(query)`:
1. `key` = 소문자 + 공백을 한 칸으로 정리한 검색어
2. 같은 `key` 를 **600초 안에** 찾은 적이 있으면 저장한 결과의 **복사본**을 돌려줍니다(`fetch` 를 부르지 않음)
3. 막힌 뒤 **300초가 지나지 않았으면** `Blocked('쉬는 중')` 을 던집니다(`fetch` 를 부르지 않음)
4. `fetch` 를 부릅니다. `Blocked` 가 나면 *지금+300초* 까지 쉬도록 적고 다시 던집니다
5. `(시각, 결과)` 를 저장하고 복사본을 돌려줍니다 — 부른 쪽이 목록을 고쳐도 저장본은 그대로여야 합니다

실제 코드: `jarvis/webread.py` 의 `WebReader.search`

In [ ]:
class Blocked(Exception):
    pass

class SearchCache:
    def __init__(self, fetch, clock):
        # TODO: 저장소(사전)와 '언제까지 쉴지' 를 준비하세요.
        raise NotImplementedError('직접 구현해 보세요')

    def search(self, query: str) -> list:
        # TODO
        raise NotImplementedError('직접 구현해 보세요')

In [ ]:
class _Clock:
    def __init__(self): self.now = 1000.0
    def __call__(self): return self.now

class _Engine:
    def __init__(self): self.sent, self.block = [], False
    def __call__(self, query):
        self.sent.append(query)
        if self.block:
            raise Blocked('자동 요청으로 막음')
        return [f'결과: {query}']

def _cache_hits():
    clock, engine = _Clock(), _Engine(); cache = SearchCache(engine, clock)
    cache.search('Qwen3.8 release'); cache.search('  qwen3.8   RELEASE ')
    first = len(engine.sent) == 1
    clock.now += 601; cache.search('qwen3.8 release')
    return first and len(engine.sent) == 2

def _backoff():
    clock, engine = _Clock(), _Engine(); cache = SearchCache(engine, clock)
    engine.block = True
    try:
        cache.search('a'); return False
    except Blocked:
        pass
    try:
        cache.search('b'); return False
    except Blocked:
        pass
    if len(engine.sent) != 1:
        return False            # 쉬는 동안에는 보내지 않아야 합니다
    engine.block = False; clock.now += 301
    return cache.search('b') == ['결과: b']

def _copies():
    clock, engine = _Clock(), _Engine(); cache = SearchCache(engine, clock)
    cache.search('x').clear()
    return cache.search('x') == ['결과: x']

check('11. 검색 캐시', [
    ('같은 검색어는 10분 동안 다시 보내지 않는다', _cache_hits),
    ('막히면 5분 쉬고, 그 뒤엔 다시 찾는다', _backoff),
    ('복사본을 돌려준다', _copies),
])

### 12. 도구 호출 루프 ★★★ (ADR-056)

모델이 도구(`web_search`)를 부르면 앱이 실행하고, 결과를 대화에 붙여 **다시 묻습니다**.
**실제 사건:** 처음엔 마지막 차례에 도구를 빼서 답을 받으려 했는데, gpt-oss 는 도구를 뺀 차례에도 도구를 부르며 본문을 비웠습니다. 그래서 도구는 계속 주고 **차례 수만** 끊습니다.

`run_tool_loop(model, runner, max_rounds=3)`:
- `conversation = []` 을 만듭니다(도구 결과를 쌓는 목록).
- `round_number` 를 0 부터 `max_rounds` 까지(**max_rounds + 1 번**) 돌며:
  - `content, calls = model(conversation)` — `calls` 는 `[(이름, 인자사전), ...]`
  - `calls` 가 비어 있으면 `content` 를 돌려줍니다(답을 썼다)
  - `round_number == max_rounds` 이면 반복을 끝냅니다(더는 도구를 실행해 주지 않음)
  - 아니면 각 `(name, args)` 마다 `runner(name, args)` 의 결과를 `conversation` 에 `('tool', name, 결과)` 로 붙입니다
- 반복이 끝났는데 `content` 가 있으면 `content`, 없으면 `RuntimeError('검색만 되풀이')`

실제 코드: `jarvis/providers.py` 의 `_generate_with_tools`

In [ ]:
def run_tool_loop(model, runner, max_rounds: int = 3) -> str:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
def _scripted(*turns):
    """차례마다 (content, calls) 를 내는 가짜 모델. 받은 대화를 기록합니다."""
    seen, script = [], list(turns)
    def model(conversation):
        seen.append(list(conversation))
        return script.pop(0)
    return model, seen

def _one_search():
    asked = []
    model, seen = _scripted(('', [('web_search', {'query': 'qwen3.8 공개일'})]), ('8월입니다', []))
    answer = run_tool_loop(model, lambda n, a: asked.append(a['query']) or '자료: 2026년 8월')
    return answer == '8월입니다' and asked == ['qwen3.8 공개일'] and seen[1] == [('tool', 'web_search', '자료: 2026년 8월')]

def _keeps_calling():
    asked = []
    model, _ = _scripted(*[('', [('web_search', {'query': 'q'})])] * 4)
    try:
        run_tool_loop(model, lambda n, a: asked.append(n) or '자료', max_rounds=3)
        return False
    except RuntimeError:
        return len(asked) == 3

def _text_at_limit():
    model, _ = _scripted(*[('', [('web_search', {'query': 'q'})])] * 3, ('답을 썼습니다', [('web_search', {'query': 'q'})]))
    return run_tool_loop(model, lambda n, a: '자료', max_rounds=3) == '답을 썼습니다'

check('12. 도구 호출 루프', [
    ('도구 없이 바로 답', lambda: run_tool_loop(_scripted(('바로 답', []))[0], lambda n, a: '') == '바로 답'),
    ('한 번 찾고 답', _one_search),
    ('도구만 되풀이하면 끊는다', _keeps_calling),
    ('한도에서 글이 있으면 그 글', _text_at_limit),
])

### 13. 코드 지문 — 옛 코드 서버 알아채기 ★★ (ADR-049)

**실제 사건:** 새 기능을 넣고 런처를 다시 켰는데, 예전부터 떠 있던 서버를 그대로 써서 새 기능이 빠져 있었습니다. 그래서 런처는 서버가 불러온 코드의 **내용 지문**과 디스크의 지문을 비교합니다. 수정 시각이 아니라 **내용**으로 잽니다 — 압축을 풀거나 복사하면 시각만 바뀌니까요.

`fingerprint(files)`: `files` 는 `{상대경로: 내용(bytes)}`.
1. `digest = hashlib.sha256()`
2. 경로를 **정렬한 순서로** 각 파일마다 `digest.update(경로.encode('utf-8') + b'\0')` 와 `digest.update(hashlib.sha256(내용).digest())`
3. `digest.hexdigest()` 의 앞 16글자

실제 코드: `jarvis/launcher.py` 의 `code_fingerprint`

In [ ]:
def fingerprint(files: dict) -> str:
    # TODO
    raise NotImplementedError('직접 구현해 보세요')

In [ ]:
_A = {'core.py': b'x = 1', 'static/app.js': b'let a;'}
check('13. 코드 지문', [
    ('길이 16', lambda: len(fingerprint(_A)) == 16),
    ('같은 내용이면 같은 지문', lambda: fingerprint(_A) == fingerprint(dict(_A))),
    ('넣은 순서는 상관없다', lambda: fingerprint(_A) == fingerprint({'static/app.js': b'let a;', 'core.py': b'x = 1'})),
    ('내용이 바뀌면 달라진다', lambda: fingerprint(_A) != fingerprint({**_A, 'core.py': b'x = 2'})),
    ('파일 이름이 바뀌어도 달라진다', lambda: fingerprint(_A) != fingerprint({'core2.py': b'x = 1', 'static/app.js': b'let a;'})),
])

In [ ]:
# 관찰: 지금 디스크의 ALEX 코드 지문. 떠 있는 서버의 지문(/health 의 code)과 다르면 런처가 '앱 종료 후 다시 실행' 을 안내합니다.
def _look():
    from jarvis.launcher import code_fingerprint
    print('디스크의 코드 지문:', code_fingerprint())
observe('관찰 13', _look)

---
## 5부. 왜 그렇게 만들었을까 — 개념 확인 ★

각 질문의 답을 `'A'`, `'B'`, `'C'` 중 하나로 `answers` 에 적고 검사 셀을 실행하세요.

**q1.** FA 끄기를 데스크톱에만 적용한 이유는?
- A. 노트북은 GPU 가 느려서
- B. 노트북은 FA 를 끄고 잰 적이 없어서 — 재지 않은 조건으로 바꾸지 않으려고
- C. 노트북에는 Ollama 가 없어서

**q2.** 사용자가 바로잡은 것은 approved, ALEX 가 웹에서 찾은 사실은 candidate 로 두는 이유는?
- A. 찾은 사실은 틀릴 수 있어서 — 대화 참고(교훈)로는 쓰되, 가중치 학습 후보는 사람이 승인해야 한다
- B. candidate 가 더 빨라서
- C. 웹 자료는 90일 뒤 자동으로 approved 가 되니까

**q3.** 모델을 비교하는 측정에서 교훈(배운 것)을 끄는 이유는?
- A. 측정이 빨라져서
- B. 교훈이 답을 도와주면 모델 자체 실력이 부풀려 보여서
- C. 교훈이 늘 틀려서

**q4.** tok/s(초당 토큰)만 보고 모델을 고르면 안 되는 이유는?
- A. 처리량은 대기시간이 아니다 — 모델마다 생각에 쓰는 토큰 수가 달라 과제당 걸리는 시간이 다르다
- B. tok/s 는 잴 수 없어서
- C. 큰 모델은 항상 tok/s 가 높아서

**q5.** 문맥 창 8192 에서 빈 답이 난 이유는?
- A. 모델 파일이 깨져서
- B. 긴 대화가 창을 거의 채워, 남은 자리를 생각(thinking)에 다 써서
- C. 인터넷이 끊겨서

**q6.** 이 PC 에서 밀집형 27B 가 MoE 122B 보다 빠르지 않은 이유는?
- A. 밀집 모델은 매 토큰 모든 파라미터를 쓰는데, VRAM 을 넘쳐 RAM 으로 간 부분이 병목이 되어서
- B. 27B 가 더 최신이어서
- C. 122B 는 GPU 를 두 개 써서

In [ ]:
answers = {'q1': '', 'q2': '', 'q3': '', 'q4': '', 'q5': '', 'q6': ''}   # TODO: 'A' / 'B' / 'C'

In [ ]:
_KEY = {"q1": "96e2e2c94d", "q2": "3d9c97ee45", "q3": "74fe5019ed", "q4": "0eca085218", "q5": "fbb43cbbf5", "q6": "ac78c64558"}   # 정답은 해시로만 둡니다
def _right(q):
    if not answers[q]:
        raise NotImplementedError
    return hashlib.sha256(f"{q}:{answers[q].strip().upper()}".encode()).hexdigest()[:10] == _KEY[q]
check('14. 개념 확인', [(q, (lambda q=q: _right(q))) for q in sorted(_KEY)])

---
## 다 풀었다면

| 문제 | 실제 ALEX 코드 | 관련 기록 |
|---|---|---|
| 1 | `jarvis/launcher.py` `ollama_environment` | DECISIONS ADR-054 |
| 2, 5, 6, 9, 10 | `jarvis/lessons.py` | ADR-055, ADR-057 |
| 3 | `jarvis/providers.py` `_raise_if_exhausted` | PERFORMANCE 26-1 |
| 4, 8 | `jarvis/core.py` `_tool_source`, `PRIVATE_QUERY` | ADR-056 |
| 7 | `jarvis/lessons.py` `looks_like_correction` | ADR-057 |
| 11 | `jarvis/webread.py` `WebReader.search` | ADR-056 |
| 12 | `jarvis/providers.py` `_generate_with_tools` | ADR-056 |
| 13 | `jarvis/launcher.py` `code_fingerprint` | ADR-049 |

**ALEX 와 같이 공부하기:** 막히면 ALEX 에 "이 함수의 실패 원인을 설명하고 힌트 하나만 주세요. 완성 코드는 아직 주지 마세요" 라고 물으세요. 다 푼 뒤에는 "이 함수의 경계 조건 2개를 더 제안해 주세요" 라고 물어 검사를 늘려 보세요. **ALEX 가 준 코드도 검사 셀로 확인하세요.**